In [2]:
# Library imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path
from collections import Counter
from collections import defaultdict
from scipy.stats import wilcoxon
from scipy import stats

# Visualization settings
plt.rcParams['figure.figsize'] = (12, 8)
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Display the dataframe to fit nicely on the screen
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 500)

In [3]:
# Define the directory containing experiment results
dir_experiments = "../experiments/nimages_filterReduction/results"

# Dictionary to store result file paths for each superpixel/nimage combination
results_files = defaultdict(dict)

# Iterate through each superpixel folder in the results directory
for superpixel_folder in sorted(os.listdir(dir_experiments)):
    superpixel_path = os.path.join(dir_experiments, superpixel_folder)
    # Check if the folder is a valid superpixel folder
    if os.path.isdir(superpixel_path) and superpixel_folder.startswith("super"):
        # Store result files (.csv) found in the folder
        for file in sorted(os.listdir(superpixel_path)):
            if file.endswith("_results.csv") or file.endswith("_layer3_test1-classified-images.cvs"):
                results_files[superpixel_folder][file] = os.path.join(superpixel_path, file)

# List to store extracted results and nimage values
results_data = []
nimages_values = [1]  # Start with 1 image as default

# Extract metrics from each result file, adapting for folders containing 'filterReduction'
for folder, contents in results_files.items():
    # Parse superpixel and nimage values from folder name
    # Example folder name: super100_images5_filterReduction
    parts = folder.split("_")
    superpixel = int(parts[0].replace('super', ''))
    nimage = int(parts[1].replace('images', ''))
    nimages_values.append(nimage)
    for seed, file in contents.items():
        if isinstance(file, str) and file.endswith('_results.csv'):
            results_file = file
            seed_value = int(seed.replace('_results.csv', '').replace('seed', ''))
            with open(results_file, 'r') as f:
                lines = f.readlines()
                if len(lines) >= 4:
                    # Extract accuracy and kappa metrics from the first two lines
                    class1_accuracy, class2_accuracy = map(float, lines[0].strip().split(';')[:2])
                    kappa, global_accuracy = map(float, lines[1].strip().split(';')[:2])
                    # Try to extract nfeat from line 3 or 5 if available
                    if len(lines) > 5 and ': ' in lines[5]:
                        try:
                            nfeat = int(lines[5].strip().split(': ')[1])
                        except (IndexError, ValueError):
                            nfeat = 0
                    elif len(lines) > 3 and ': ' in lines[3]:
                        try:
                            nfeat = int(lines[3].strip().split(': ')[1])
                        except (IndexError, ValueError):
                            nfeat = 0
                    # Append the extracted metrics to the results list
                    results_data.append({
                        'superpixel': superpixel,
                        'nimage': nimage,
                        'seed': seed_value,
                        'filterReduction': 'filterReduction' in folder,
                        'class1_accuracy': class1_accuracy,
                        'class2_accuracy': class2_accuracy,
                        'kappa': kappa,
                        'global_accuracy': global_accuracy,
                        'nfeat': nfeat
                    })

# Convert the results list to a DataFrame
df_nimage = pd.DataFrame(results_data)

df_nimage.head(100)

# Save the DataFrame to a CSV file for further analysis
# df_nimage.to_csv('nimages_results_summary.csv', index=False)

,superpixel,nimage,seed,filterReduction,class1_accuracy,class2_accuracy,kappa,global_accuracy,nfeat
0,50,2,1011,False,0.874439,0.984355,0.865427,0.970404,105800
1,50,2,1213,False,0.892377,0.986962,0.886130,0.974957,105800
2,50,2,123,False,0.892377,0.991525,0.902934,0.978941,105800
3,50,2,2735,False,0.852018,0.986310,0.858058,0.969266,105800
4,50,2,42,False,0.883408,0.983051,0.866459,0.970404,105800
...,...,...,...,...,...,...,...,...,...
75,50,5,456,True,0.668161,0.951760,0.619921,0.915766,21160
76,50,5,6854,True,0.807175,0.980443,0.807739,0.958452,21160
77,50,5,7580,True,0.807175,0.977184,0.796569,0.955606,21160
78,50,5,789,True,0.856502,0.971969,0.811368,0.957314,21160


In [ ]:
# Define the metrics to analyze
metrics = ['class1_accuracy', 'class2_accuracy', 'kappa', 'global_accuracy', 'nfeat']

# Group results by number of superpixels and calculate mean and standard deviation for each metric
summary_stats = df_nimage.groupby(['nimage', 'filterReduction'])[metrics[:]].agg(['mean', 'std']).reset_index()

# Rename columns for easier access (e.g., 'kappa_mean', 'global_accuracy_mean')
summary_stats.columns = ['_'.join(col).strip() for col in summary_stats.columns.values]

# # Select only the metrics of interest for visualization and highlight the highest values
# summary_stats = summary_stats[['superpixel_', 'filterReduction_', 'class1_accuracy_mean', 'class1_accuracy_std', 'class2_accuracy_mean', 'class2_accuracy_std', 'kappa_mean', 'kappa_std', 'global_accuracy_mean', 'global_accuracy_std']].style.highlight_max(
#     subset=['class1_accuracy_mean', 'class2_accuracy_mean', 'kappa_mean', 'global_accuracy_mean', 'global_accuracy_std'], color='gray'
# )

# # Format values as percentages for better presentation
# summary_stats = summary_stats.format({'kappa_mean': '{:.4%}', 'global_accuracy_mean': '{:.4%}'})

# Mostrar apenas algumas colunas específicas
summary_stats[['nimage_', 'filterReduction_', 'class1_accuracy_mean', 'class1_accuracy_std', 'class2_accuracy_mean', 'class2_accuracy_std', 'kappa_mean', 'kappa_std', 'global_accuracy_mean', 'global_accuracy_std', 'nfeat_mean']]

# # Renomeia as colunas para facilitar o acesso
# summary_stats_df.columns = ['superpixel', 'filterReduction'] + [f"{m}_{stat}" for m in metrics[:-1] for stat in ['mean', 'std']]

# # Ordena pelo número de superpixels e filterReduction
# summary_stats_df = summary_stats_df.sort_values(['superpixel', 'filterReduction']).reset_index(drop=True)

# summary_stats_df    

# # Destaca os maiores valores das métricas principais
# styled_summary = summary_stats_df.style.highlight_max(
#     subset=['class1_accuracy_mean', 'class2_accuracy_mean', 'kappa_mean', 'global_accuracy_mean', 'global_accuracy_std'],
#     color='gray'
# ).format({'kappa_mean': '{:.4%}', 'global_accuracy_mean': '{:.4%}'})

# styled_summary


# Display the styled DataFrame
# summary_stats

# Export the styled summary table to LaTeX code (without styling)
# latex_table = df_nsuper.groupby('superpixel')[metrics[:-1]].agg(['mean', 'std']).loc[superpixels_values].to_latex(float_format="%.4f")
# print(latex_table)

,nimage_,filterReduction_,class1_accuracy_mean,class2_accuracy_mean,kappa_mean,global_accuracy_mean,nfeat_mean
0,2,False,0.879821,0.986245,0.875569,0.972738,105800.0
1,2,True,0.793274,0.971773,0.770263,0.949118,21160.0
2,3,False,0.887444,0.985984,0.879518,0.973477,158700.0
3,3,True,0.803588,0.972425,0.779975,0.950996,21160.0
4,4,False,0.894619,0.986050,0.884212,0.974445,211600.0
5,4,True,0.795067,0.978683,0.793463,0.955378,21160.0
6,5,False,0.897309,0.986375,0.887052,0.975071,264500.0
7,5,True,0.804484,0.971904,0.777784,0.950655,21160.0


In [ ]:
wilcoxon_results = pd.DataFrame(columns=['nimage1', 'nimage2', 'statistic', 'p_value'])

df_reduc = df_nimage[df_nimage['filterReduction'] == True]
nimages_unique = sorted(df_reduc['nimage'].unique())
for i in range(len(nimages_unique)):
    for j in range(i + 1, len(nimages_unique)):
        n1 = nimages_unique[i]
        n2 = nimages_unique[j]
        
        data_n1 = df_reduc[df_reduc['nimage'] == n1]['global_accuracy']
        data_n2 = df_reduc[df_reduc['nimage'] == n2]['global_accuracy']
        
        if len(data_n1) == len(data_n2) and len(data_n1) > 0:
            statistic, p_value = wilcoxon(data_n1, data_n2)
            wilcoxon_results.loc[len(wilcoxon_results)] = {
                'nimage1': n1,
                'nimage2': n2,
                'statistic': statistic,
                'p_value': p_value
            }

print("Resultados do teste de Wilcoxon entre nimages (filterReduction=True):")
print(wilcoxon_results)

significant = wilcoxon_results[wilcoxon_results['p_value'] < 0.05]
print("Comparações com diferença estatisticamente significativa:")
print(significant)
print(f"Total de comparações significativas: {len(significant)} de {len(wilcoxon_results)}")

Resultados do teste de Wilcoxon entre nimages (filterReduction=True):
   nimage1  nimage2  statistic   p_value
0        2        3       26.0  0.921875
1        2        4       21.0  0.556641
2        2        5       21.0  0.910156
3        3        4       23.0  0.695312
4        3        5       24.0  0.769531
5        4        5       16.0  0.496094
Comparações com diferença estatisticamente significativa:
Empty DataFrame
Columns: [nimage1, nimage2, statistic, p_value]
Index: []
Total de comparações significativas: 0 de 6


In [9]:
wilcoxon_results = pd.DataFrame(columns=['nimage1', 'nimage2', 'statistic', 'p_value'])

df_reduc = df_nimage[df_nimage['filterReduction'] == False]
nimages_unique = sorted(df_reduc['nimage'].unique())
for i in range(len(nimages_unique)):
    for j in range(i + 1, len(nimages_unique)):
        n1 = nimages_unique[i]
        n2 = nimages_unique[j]
        
        data_n1 = df_reduc[df_reduc['nimage'] == n1]['global_accuracy']
        data_n2 = df_reduc[df_reduc['nimage'] == n2]['global_accuracy']
        
        if len(data_n1) == len(data_n2) and len(data_n1) > 0:
            statistic, p_value = wilcoxon(data_n1, data_n2)
            wilcoxon_results.loc[len(wilcoxon_results)] = {
                'nimage1': n1,
                'nimage2': n2,
                'statistic': statistic,
                'p_value': p_value
            }

print("Resultados do teste de Wilcoxon entre nimages (filterReduction=False):")
print(wilcoxon_results)

significant = wilcoxon_results[wilcoxon_results['p_value'] < 0.05]
print("Comparações com diferença estatisticamente significativa:")
print(significant)
print(f"Total de comparações significativas: {len(significant)} de {len(wilcoxon_results)}")


Resultados do teste de Wilcoxon entre nimages (filterReduction=False):
   nimage1  nimage2  statistic   p_value
0        2        3       21.5  0.574219
1        2        4       11.0  0.101562
2        2        5        7.5  0.042969
3        3        4       15.0  0.425781
4        3        5        8.0  0.097656
5        4        5       18.0  0.359375
Comparações com diferença estatisticamente significativa:
   nimage1  nimage2  statistic   p_value
2        2        5        7.5  0.042969
Total de comparações significativas: 1 de 6
